# 🧪 Химический Классификатор (PERFECT FINAL EDITION)
---
Ноутбук с атомарной структурой и **полной двухэтапной стерилизацией** категорий.

In [ ]:
%%capture
!pip install pandas numpy catboost requests tqdm openpyxl

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
import requests
import json
import re
import os
from tqdm import tqdm

## 📂 Настройка файлов и LLM

In [ ]:
INPUT_FILE = r'data/fragrances_input.xlsx'
TEACHER_FILE = 'Teacher_Qwen_2000.xlsx'
HYBRID_FILE = r'output/hybrid_result.xlsx'
PERFECT_FILE = r'output/PERFECT_FINAL_7000.xlsx'

DESC_COL = 'Описание 350 крт'
TARGET_COL = 'Продукт единый'

In [ ]:
LLM_URL = "http://127.0.0.1:1234/v1/chat/completions"
LLM_MODEL = "qwen2.5-7b-instruct"
SYSTEM_PROMPT = """Ты — эксперт-аналитик химического рынка. Классифицируй товары по списку.
Верни строго JSON с одним полем: "Продукт единый".

СПИСОК КАТЕГОРИЙ (выбирай СТРОГО из него):
- Ароматизаторы пищевые (жидкие)
- Ароматизаторы пищевые (сухие/порошок)
- Ароматизаторы спиртосодержащие (напитки)
- Ароматизаторы коптильные (Жидкий дым)
- Ароматизаторы для табачной промышленности
- Ароматизаторы для вейпов и ЭСДН
- Ароматические капсулы/сферики (табак)
- Отдушки для парфюмерии (базы/композиции)
- Отдушки для косметики и мыла
- Отдушки для бытовой химии
- Отдушки для автохимии
- Эфирные масла (натуральные/нативные)
- Эфирные масла (реконструированные/синтетика)
- Растительные экстракты и олеорезины
- Химические изоляты (чистые в-ва: ментол, ванилин)
- Смеси душистых веществ (пром. сырье)
- Препараты для кондитерских изделий и жвачки
- Препараты для производства соусов и кетчупов
- Сырье и концентраты для напитков
- Вспомогательные в-ва (растворители/носители)
- Пищевые добавки комплексные
- Кормовые добавки (животные)
- Ароматизаторы фармацевтические
- Образцы для лабораторных исследований
- Прочее (не классифицировано)

ИНСТРУКЦИЯ:
- Слово "ОЛЕОРЕЗИН" / "OLEORESIN" / "ЭКСТРАКТ" -> "Растительные экстракты и олеорезины"
- "Сферические капсулы" -> "Ароматические капсулы/сферики (табак)"
- Для сауны и свечей -> "Отдушки для бытовой химии"
- Слова "ПИЩЕВЫЕ", "ДОБАВКА", "ДЛЯ КОНДИТЕРСКОЙ" -> НИКОГДА не относи к эфирным маслам!"""

## 🧠 Этап 1: Первичная разметка (2000 строк)

In [ ]:
def classify_row(text):
    if pd.isna(text) or text == "": return "Пусто"
    try:
        response = requests.post(LLM_URL, json={
            "model": LLM_MODEL, 
            "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Вход: {text}"}],
            "temperature": 0.0}, timeout=10)
        res_text = response.json()['choices'][0]['message']['content']
        match = re.search(r'\{.*\}', res_text, re.DOTALL)
        parsed = json.loads(match.group(0)) if match else json.loads(res_text)
        return parsed.get(TARGET_COL, "Ошибка")
    except: return "Ошибка"

In [ ]:
df_full = pd.read_excel(INPUT_FILE)
df_llm = df_full[df_full[TARGET_COL].isna()].sample(2000).copy()

for i, idx in enumerate(tqdm(df_llm.index)):
    if pd.notna(df_llm.at[idx, TARGET_COL]): continue
    df_llm.at[idx, TARGET_COL] = classify_row(df_llm.at[idx, DESC_COL])
    if (i + 1) % 100 == 0: df_llm.to_excel(TEACHER_FILE, index=False)

df_llm[TARGET_COL] = df_llm[TARGET_COL].astype(str).str.replace(r'[^а-яА-Яa-zA-Z0-9\(\)\/\-\s:]', '', regex=True)
df_llm.to_excel(TEACHER_FILE, index=False)

## 🩺 Этап 2: Санитарная очистка (fix_qwen_errors)

In [ ]:
def fix_qwen_errors(val):
    if pd.isna(val): return "Прочее (не классифицировано)"
    val_str = str(val).strip()
    if "пром сырье" in val_str or "промышленное сырье" in val_str: return "Смеси душистых веществ (пром. сырье)"
    if "ментол ванилин" in val_str or "чистые в-ва" in val_str: return "Химические изоляты (чистые в-ва: ментол, ванилин)"
    
    val_lower = val_str.lower()
    if "косметики и мыла" in val_lower: return "Отдушки для косметики и мыла"
    if "парфюмерии" in val_lower: return "Отдушки для парфюмерии (базы/композиции)"
    if "табак" in val_lower or "сигарет" in val_lower: return "Ароматизаторы для табачной промышленности"
    if "корм" in val_lower: return "Кормовые добавки (животные)"
    if "порошок" in val_lower or "сухие" in val_lower: return "Ароматизаторы пищевые (сухие/порошок)"
    if "капсулы" in val_lower or "сферики" in val_lower: return "Ароматические капсулы/сферики (табак)"
    return val_str

df_llm[TARGET_COL] = df_llm[TARGET_COL].apply(fix_qwen_errors)
df_llm.to_excel(TEACHER_FILE, index=False)

## 🚀 Этап 3: CatBoost (Массовая разметка)

In [ ]:
df_train = pd.read_excel(TEACHER_FILE).dropna(subset=[DESC_COL, TARGET_COL])
model = CatBoostClassifier(iterations=300, learning_rate=0.1, text_features=[DESC_COL], random_seed=42, verbose=0)
model.fit(df_train[[DESC_COL]], df_train[TARGET_COL])

df_predict = df_full.iloc[len(df_train):].copy()
probs = np.max(model.predict_proba(df_predict[[DESC_COL]]), axis=1)
preds = model.predict(df_predict[[DESC_COL]])

df_predict[TARGET_COL] = [preds[i][0] if probs[i] >= 0.70 else "ТРЕБУЕТСЯ LLM" for i in range(len(preds))]
df_result = pd.concat([df_train, df_predict], ignore_index=True)
df_result.to_excel(HYBRID_FILE, index=False)

## 🧠 Этап 4: Доразметка (LLM Phase 2)

In [ ]:
def classify_row_llm(text):
    try:
        resp = requests.post(LLM_URL, json={"model": LLM_MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Вход: {text}"}], "temperature": 0.0}, timeout=15)
        res = resp.json()['choices'][0]['message']['content']
        m = re.search(r'\{.*\}', res, re.DOTALL)
        p = json.loads(m.group(0)) if m else json.loads(res)
        val = p.get(TARGET_COL, "Ошибка")
        if isinstance(val, list): val = val[0] if val else "Ошибка"
        return str(val).strip()
    except: return "Ошибка"

mask = df_result[TARGET_COL] == "ТРЕБУЕТСЯ LLM"
for i, idx in enumerate(tqdm(df_result[mask].index)):
    df_result.at[idx, TARGET_COL] = classify_row_llm(df_result.at[idx, DESC_COL])
    if (i + 1) % 50 == 0: df_result.to_excel(HYBRID_FILE, index=False)

## 💎 Этап 5: Финальная стерилизация (clean_final_garbage)

In [ ]:
# Словарь жесткого маппинга (Порядок важен!)
STEWARD_MAPPING = {
    "сухие/порошок": "Ароматизаторы пищевые (сухие/порошок)",
    "реконструированные/синтетика": "Эфирные масла (реконструированные/синтетика)",
    "натуральные/нативные": "Эфирные масла (натуральные/нативные)",
    "нативные": "Эфирные масла (натуральные/нативные)",
    "мыловар": "Отдушки для косметики и мыла",
    "косметики и мыла": "Отдушки для косметики и мыла",
    "парфюмерии": "Отдушки для парфюмерии (базы/композиции)",
    "бытовой химии": "Отдушки для бытовой химии",
    "автохимии": "Отдушки для автохимии",
    "вейп": "Ароматизаторы для вейпов и ЭСДН",
    "напитков": "Сырье и концентраты для напитков",
    "табач": "Ароматизаторы для табачной промышленности",
    "коптиль": "Ароматизаторы коптильные (Жидкий дым)",
    "фармацевт": "Ароматизаторы фармацевтические",
    "капсулы": "Ароматические капсулы/сферики (табак)",
    "сферики": "Ароматические капсулы/сферики (табак)",
    "корм": "Кормовые добавки (животные)",
    "добавки комплексные": "Пищевые добавки комплексные",
    "кондитер": "Препараты для кондитерских изделий и жвачки",
    "соусов": "Препараты для производства соусов и кетчупов",
    "кетчуп": "Препараты для производства соусов и кетчупов",
    "экстракты": "Растительные экстракты и олеорезины",
    "олеорезин": "Растительные экстракты and олеорезины",
    "изоляты": "Химические изоляты (чистые в-ва: ментол, ванилин)",
    "ментол": "Химические изоляты (чистые в-ва: ментол, ванилин)",
    "пром. сырье": "Смеси душистых веществ (пром. сырье)",
    "промышленное сырье": "Смеси душистых веществ (пром. сырье)",
    "жидкие": "Ароматизаторы пищевые (жидкие)",
    "пищевых продуктов": "Ароматизаторы пищевые (жидкие)",
    "ароматизаторы пищевые": "Ароматизаторы пищевые (жидкие)",
    "смеси душистых веществ": "Смеси душистых веществ (пром. сырье)"
}

In [ ]:
def clean_final_garbage(val):
    if pd.isna(val): return "Прочее (не классифицировано)"
    val_str = str(val).strip().lower()
    
    for key, correct_cat in STEWARD_MAPPING.items():
        if key in val_str:
            return correct_cat
            
    return "Прочее (не классифицировано)"

print("Запускаю финальную стерилизацию категорий...")
df_result[TARGET_COL] = df_result[TARGET_COL].apply(clean_final_garbage)

os.makedirs(os.path.dirname(PERFECT_FILE), exist_ok=True)
df_result.to_excel(PERFECT_FILE, index=False)

print("\n=== ТОП-10 ИТОГОВЫХ КАТЕГОРИЙ ===")
print(df_result[TARGET_COL].value_counts().head(10))
print(f"\nВсе данные очищены и сохранены в файл: {PERFECT_FILE}")